# Datos del equipo (COMPLETAR)

Identificador de grupo: Grupo G2.

| Nombre y apellido | % de contribución |
|-------------------|-------------------|
|Arantzazu Otal Alberro|     20%        |
|                   |                   |
|                   |                   |
|                   |                   |
|                   |                   |


## 1. CONFIGURACIÓN INICIAL Y CARGA DE LIBRERÍAS

In [20]:
# Imports necesarios
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix, roc_auc_score, classification_report
)

# AIF360
import aif360
from aif360.datasets import CompasDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from aif360.algorithms.preprocessing import Reweighing
from aif360.algorithms.preprocessing import DisparateImpactRemover
from aif360.algorithms.inprocessing import AdversarialDebiasing
from aif360.algorithms.postprocessing import EqOddsPostprocessing

# Fairlearn
import fairlearn
from fairlearn.metrics import (
    MetricFrame, selection_rate, demographic_parity_difference,
    equalized_odds_difference
)
from fairlearn.reductions import ExponentiatedGradient, DemographicParity, EqualizedOdds
from fairlearn.postprocessing import ThresholdOptimizer

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)

print("✓ Librerías cargadas correctamente")
print(f"AIF360 version: {aif360.__version__}")
print(f"Fairlearn version: {fairlearn.__version__}")


✓ Librerías cargadas correctamente
AIF360 version: 0.6.1
Fairlearn version: 0.14.0


## 2. CARGA Y EXPLORACIÓN DEL DATASET COMPAS

El dataset COMPAS contiene información sobre acusados en el condado de Broward, Florida, entre 2013-2014, incluyendo su puntuación de riesgo de reincidencia y si reincidieron dentro de 2 años.

In [7]:
# Descargar el dataset COMPAS si no está ya descargado
!wget -nv -O ~/miniconda3/envs/ia_confiable/lib/python3.10/site-packages/aif360/data/raw/compas/compas-scores-two-years.csv https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv

2026-10-19 19:15:08 URL:https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv [2546489/2546489] -> "/home/arantxa/miniconda3/envs/ia_confiable/lib/python3.10/site-packages/aif360/data/raw/compas/compas-scores-two-years.csv" [1]


In [ ]:
from aif360.datasets import CompasDataset

# Cargar dataset COMPAS usando AIF360
dataset_orig = CompasDataset(
    protected_attribute_names=['race'],
    privileged_classes=[['Caucasian']],
    categorical_features=['sex', 'age_cat', 'c_charge_degree', 'c_charge_desc'],
    features_to_keep=[
        'sex', 'age', 'age_cat', 'race',
        'juv_fel_count', 'juv_misd_count', 'juv_other_count',
        'priors_count', 'c_charge_degree', 'c_charge_desc',
        'two_year_recid'
    ]
)

# Información básica del dataset
print(f"Número total de instancias: {dataset_orig.features.shape[0]}")
print(f"Número de features: {dataset_orig.features.shape[1]}")
print(f"Nombres de features: {dataset_orig.feature_names}")
print(f"Atributo protegido: {dataset_orig.protected_attribute_names}")
print(f"Grupo privilegiado: Caucasian")
print(f"Grupo no privilegiado: African-American")


NameError: name 'CompasDataset' is not defined

In [ ]:
# Convertir a DataFrame para análisis exploratorio
df = pd.DataFrame(
    data=dataset_orig.features,
    columns=dataset_orig.feature_names
)
df['recidivism'] = dataset_orig.labels.ravel()
df['race_binary'] = dataset_orig.protected_attributes.ravel()

# Mapear valores para mejor interpretación
df['race_label'] = df['race_binary'].map({1: 'Caucasian', 0: 'African-American'})

print("Primeras filas del dataset:")
df.head(10)


### 2.1 Análisis de Distribución Demográfica

In [ ]:
# Distribución por raza
print("=== DISTRIBUCIÓN POR RAZA ===")
race_distribution = df['race_label'].value_counts()

# EJERCICIO 1: Completa el código para imprimir la distribución racial del dataset y los porcentajes.
# === INICIO SOLUCIÓN ===

# === FIN SOLUCIÓN ===

In [ ]:
# Tasa de reincidencia por raza
print("=== TASA DE REINCIDENCIA POR RAZA ===")

# EJERCICIO 2: Calcula la tasa de reincidencia para cada grupo racial e imprime la diferencia absoluta.
# === INICIO SOLUCIÓN ===
# recidivism_by_race =
# === FIN SOLUCIÓN ===

In [ ]:
# Visualización de distribuciones
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Distribución racial
race_distribution.plot(kind='bar', ax=axes[0], color=['#2E86AB', '#A23B72'])
axes[0].set_title('Distribución por Raza')
axes[0].set_xlabel('Raza')
axes[0].set_ylabel('Número de Individuos')
axes[0].tick_params(axis='x', rotation=45)

# Tasa de reincidencia por raza
recidivism_by_race.plot(kind='bar', ax=axes[1], color=['#2E86AB', '#A23B72'])
axes[1].set_title('Tasa de Reincidencia por Raza')
axes[1].set_xlabel('Raza')
axes[1].set_ylabel('Proporción de Reincidencia')
axes[1].axhline(y=df['recidivism'].mean(), color='red', linestyle='--',
               label=f'Media global: {df["recidivism"].mean():.2f}')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

# Distribución de edad por raza
for race in df['race_label'].unique():
    subset = df[df['race_label'] == race]
    axes[2].hist(subset['age'], alpha=0.5, label=race, bins=20)
axes[2].set_title('Distribución de Edad por Raza')
axes[2].set_xlabel('Edad')
axes[2].set_ylabel('Frecuencia')
axes[2].legend()

plt.tight_layout()
plt.show()


 ### 2.2 Análisis de Correlaciones y Features

In [ ]:
# Análisis de correlaciones con el outcome y el atributo protegido
numeric_cols = ['age', 'juv_fel_count', 'juv_misd_count',
                'juv_other_count', 'priors_count']

correlation_data = df[numeric_cols + ['recidivism', 'race_binary']].copy()

# Matriz de correlación
plt.figure(figsize=(10, 8))
correlation_matrix = correlation_data.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1)
plt.title('Matriz de Correlación - Dataset COMPAS')
plt.tight_layout()
plt.show()

print("Correlaciones con reincidencia:")
print(correlation_matrix['recidivism'].sort_values(ascending=False))
print("\nCorrelaciones con raza:")
print(correlation_matrix['race_binary'].sort_values(ascending=False))


## 3. PREPARACIÓN DE DATOS Y MODELO BASE

In [ ]:
# División train/test manteniendo proporciones
dataset_train, dataset_test = dataset_orig.split([0.7], shuffle=True, seed=42)

# Extraer features y labels
X_train = dataset_train.features
y_train = dataset_train.labels.ravel()
sensitive_train = dataset_train.protected_attributes.ravel()

X_test = dataset_test.features
y_test = dataset_test.labels.ravel()
sensitive_test = dataset_test.protected_attributes.ravel()

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape}")
print(f"Tamaño del conjunto de prueba: {X_test.shape}")

# Normalización de features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 3.1 Entrenamiento del Modelo Base (Sin Mitigación)

In [ ]:
# EJERCICIO 3: Entrena un modelo de regresión logística base.
# Completa el código para entrenar el modelo y obtener predicciones.
# === INICIO SOLUCIÓN ===
# Entrenar modelo base
# baseline_model =
#
# Predicciones
# y_pred_baseline =
# y_pred_proba_baseline =
# === FIN SOLUCIÓN ===

# Métricas de rendimiento general
print("=== MÉTRICAS DEL MODELO BASE ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_baseline):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_baseline):.3f}")
print(f"Recall: {recall_score(y_test, y_pred_baseline):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_baseline):.3f}")


### 3.2 Evaluación de Sesgo en el Modelo Base

In [ ]:
# Crear dataset con predicciones para métricas AIF360
dataset_pred_baseline = dataset_test.copy()
dataset_pred_baseline.labels = y_pred_baseline.reshape(-1, 1)

# Calcular métricas de equidad con AIF360
privileged_groups = [{'race': 1}]  # Caucasian
unprivileged_groups = [{'race': 0}]  # African-American

metric_baseline = ClassificationMetric(
    dataset_test,
    dataset_pred_baseline,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

print("=== MÉTRICAS DE EQUIDAD - MODELO BASE ===")
print(f"Disparate Impact: {metric_baseline.disparate_impact():.3f}")
print(f"Statistical Parity Difference: {metric_baseline.statistical_parity_difference():.3f}")
print(f"Equal Opportunity Difference: {metric_baseline.equal_opportunity_difference():.3f}")
print(f"Average Odds Difference: {metric_baseline.average_abs_odds_difference():.3f}")
print(f"Theil Index: {metric_baseline.theil_index():.3f}")


In [ ]:
# Análisis detallado por grupo usando Fairlearn
from sklearn.metrics import confusion_matrix

# Crear DataFrame con predicciones y grupos
results_df = pd.DataFrame({
    'y_true': y_test,
    'y_pred': y_pred_baseline,
    'race': sensitive_test
})

# MetricFrame para análisis estructurado
metrics = {
    'accuracy': accuracy_score,
    'precision': precision_score,
    'recall': recall_score,
    'selection_rate': selection_rate
}

mf_baseline = MetricFrame(
    metrics=metrics,
    y_true=y_test,
    y_pred=y_pred_baseline,
    sensitive_features=sensitive_test
)

print("\n=== MÉTRICAS POR GRUPO (FAIRLEARN) ===")
print(mf_baseline.by_group)
print(f"\nDiferencias entre grupos:")
print(mf_baseline.difference())

# Visualización de métricas por grupo
mf_baseline.by_group.plot(kind='bar', figsize=(10, 6))
plt.title('Métricas del Modelo Base por Grupo Racial')
plt.xlabel('Grupo (0=African-American, 1=Caucasian)')
plt.ylabel('Valor de la Métrica')
plt.legend(title='Métricas', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


 ### 3.3 Análisis de Matrices de Confusión por Grupo

In [ ]:
# EJERCICIO 4: Analiza las matrices de confusión para cada grupo racial.
# Completa el código para calcular y visualizar las matrices de confusión.
# === INICIO SOLUCIÓN ===
# Matrices de confusión por grupo
# cm_caucasian =
# cm_african_american =

# Visualización
# ...
# === FIN SOLUCIÓN ===

# EJERCICIO 5: Calcula e imprime la False Positive Rate (FPR) entre los grupos raciales.
# === INICIO SOLUCIÓN ===
# Calcular False Positive Rate para cada grupo
# fpr_caucasian =
# fpr_african_american =
# === FIN SOLUCIÓN ===

## 4. TÉCNICAS DE MITIGACIÓN DE SESGO

Ahora aplicaremos diferentes técnicas de mitigación y compararemos sus resultados.

### 4.1 Pre-processing: Reweighing

In [ ]:
# Aplicar Reweighing
RW = Reweighing(
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

# Transformar dataset de entrenamiento
dataset_train_rw = RW.fit_transform(dataset_train)

# Entrenar modelo con datos reponderados
X_train_rw = dataset_train_rw.features
y_train_rw = dataset_train_rw.labels.ravel()
weights_rw = dataset_train_rw.instance_weights.ravel()

# Normalizar features
X_train_rw_scaled = scaler.fit_transform(X_train_rw)

# Entrenar modelo
model_rw = LogisticRegression(max_iter=1000, random_state=42)
model_rw.fit(X_train_rw_scaled, y_train_rw, sample_weight=weights_rw)

# Predicciones
y_pred_rw = model_rw.predict(X_test_scaled)

# Evaluación
print("=== REWEIGHING - MÉTRICAS ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rw):.3f}")

# Métricas de equidad
dataset_pred_rw = dataset_test.copy()
dataset_pred_rw.labels = y_pred_rw.reshape(-1, 1)

metric_rw = ClassificationMetric(
    dataset_test,
    dataset_pred_rw,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

print(f"Disparate Impact: {metric_rw.disparate_impact():.3f}")
print(f"Statistical Parity Difference: {metric_rw.statistical_parity_difference():.3f}")
print(f"Equal Opportunity Difference: {metric_rw.equal_opportunity_difference():.3f}")


### 4.2 In-processing: Exponentiated Gradient con Demographic Parity

In [ ]:
from fairlearn.reductions import ExponentiatedGradient, DemographicParity

# EJERCICIO 6: Implementa mitigación usando ExponentiatedGradient con DemographicParity.
# Completa el código para entrenar el modelo con restricciones de equidad.
# === INICIO SOLUCIÓN ===
# Configurar el mitigador con DemographicParity
# mitigator =

# Entrenar con restricciones de equidad
# ...

# Predicciones
# y_pred_expgrad =
# === FIN SOLUCIÓN ===

# Evaluación
print("=== EXPONENTIATED GRADIENT - MÉTRICAS ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_expgrad):.3f}")

# Métricas con Fairlearn
mf_expgrad = MetricFrame(
    metrics={'accuracy': accuracy_score, 'selection_rate': selection_rate},
    y_true=y_test,
    y_pred=y_pred_expgrad,
    sensitive_features=sensitive_test
)

print("\nMétricas por grupo:")
print(mf_expgrad.by_group)

# Calcular Disparate Impact
sr_african = mf_expgrad.by_group.loc[0, 'selection_rate']
sr_caucasian = mf_expgrad.by_group.loc[1, 'selection_rate']
di_expgrad = sr_african / sr_caucasian if sr_caucasian > 0 else 0

print(f"\nDisparate Impact: {di_expgrad:.3f}")
print(f"Selection Rate Difference: {mf_expgrad.difference()['selection_rate']:.3f}")


### 4.3 In-processing: Exponentiated Gradient con Equalized Odds

In [ ]:
from fairlearn.reductions import EqualizedOdds

# EJERCICIO 7: Implementa ExponentiatedGradient con EqualizedOdds.
# Completa el código para aplicar la restricción de Equalized Odds.
# === INICIO SOLUCIÓN ===
# Configurar mitigador con Equalized Odds
# mitigator_eo =

# Entrenar
# ...

# Predicciones
# y_pred_eo =
# === FIN SOLUCIÓN ===

# Evaluación
print("=== EXPONENTIATED GRADIENT (EQUALIZED ODDS) - MÉTRICAS ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_eo):.3f}")

# Métricas de equidad con AIF360
dataset_pred_eo = dataset_test.copy()
dataset_pred_eo.labels = y_pred_eo.reshape(-1, 1)

metric_eo = ClassificationMetric(
    dataset_test,
    dataset_pred_eo,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

print(f"Disparate Impact: {metric_eo.disparate_impact():.3f}")
print(f"Equal Opportunity Difference: {metric_eo.equal_opportunity_difference():.3f}")
print(f"Average Odds Difference: {metric_eo.average_abs_odds_difference():.3f}")


### 4.4 Post-processing: Threshold Optimizer

In [ ]:
# Crear conjunto de validación para ajustar umbrales
X_train_sub, X_val, y_train_sub, y_val, sens_train_sub, sens_val = train_test_split(
    X_train_scaled, y_train, sensitive_train,
    test_size=0.3, random_state=42, stratify=y_train
)

# Entrenar modelo en subconjunto
model_for_threshold = LogisticRegression(max_iter=1000, random_state=42)
model_for_threshold.fit(X_train_sub, y_train_sub)

# Configurar ThresholdOptimizer
postprocessor = ThresholdOptimizer(
    estimator=model_for_threshold,
    constraints='demographic_parity',
    objective='accuracy_score',
    prefit=True
)

# Ajustar umbrales
postprocessor.fit(X_val, y_val, sensitive_features=sens_val)

# Predicciones con umbrales optimizados
y_pred_threshold = postprocessor.predict(X_test_scaled, sensitive_features=sensitive_test)

# Evaluación
print("=== THRESHOLD OPTIMIZER - MÉTRICAS ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_threshold):.3f}")

# Métricas con Fairlearn
mf_threshold = MetricFrame(
    metrics={'accuracy': accuracy_score, 'selection_rate': selection_rate},
    y_true=y_test,
    y_pred=y_pred_threshold,
    sensitive_features=sensitive_test
)

print("\nMétricas por grupo:")
print(mf_threshold.by_group)


## 5. COMPARACIÓN DE TÉCNICAS

In [ ]:
# Compilar todos los resultados
results_summary = {
    'Baseline': {
        'accuracy': accuracy_score(y_test, y_pred_baseline),
        'disparate_impact': metric_baseline.disparate_impact(),
        'stat_parity_diff': metric_baseline.statistical_parity_difference(),
        'equal_opp_diff': metric_baseline.equal_opportunity_difference()
    },
    'Reweighing': {
        'accuracy': accuracy_score(y_test, y_pred_rw),
        'disparate_impact': metric_rw.disparate_impact(),
        'stat_parity_diff': metric_rw.statistical_parity_difference(),
        'equal_opp_diff': metric_rw.equal_opportunity_difference()
    },
    'ExpGrad-DP': {
        'accuracy': accuracy_score(y_test, y_pred_expgrad),
        'disparate_impact': di_expgrad,
        'stat_parity_diff': mf_expgrad.difference()['selection_rate'],
        'equal_opp_diff': None  # No optimizado para esto
    },
    'ExpGrad-EO': {
        'accuracy': accuracy_score(y_test, y_pred_eo),
        'disparate_impact': metric_eo.disparate_impact(),
        'stat_parity_diff': metric_eo.statistical_parity_difference(),
        'equal_opp_diff': metric_eo.equal_opportunity_difference()
    },
    'ThresholdOpt': {
        'accuracy': accuracy_score(y_test, y_pred_threshold),
        'disparate_impact': sr_african / sr_caucasian if sr_caucasian > 0 else 0,
        'stat_parity_diff': mf_threshold.difference()['selection_rate'],
        'equal_opp_diff': None
    }
}

# Crear DataFrame con resultados
results_df = pd.DataFrame(results_summary).T
results_df = results_df.round(3)

print("=== COMPARACIÓN DE TODAS LAS TÉCNICAS ===")
print(results_df)


### 5.1 Visualización del compromiso Accuracy vs Equidad

In [ ]:
# EJERCICIO 8: Crea una visualización del compromiso entre accuracy y equidad.
# Completa los fragmentos de código para crear un gráfico de dispersión.
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Gráfico 1: Accuracy vs Disparate Impact
for method, metrics in results_summary.items():
    if metrics['disparate_impact'] is not None:
        # === INICIO SOLUCIÓN ===

        # === FIN SOLUCIÓN ===

axes[0].axvline(x=0.8, color='red', linestyle='--', alpha=0.5,
                label='DI Threshold (0.8)')
axes[0].axhline(y=0.65, color='gray', linestyle='--', alpha=0.5,
                label='Min Accuracy')
axes[0].fill_between([0.8, 1.2], 0.65, 1.0, alpha=0.1, color='green')
axes[0].set_xlabel('Disparate Impact', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Trade-off: Accuracy vs Disparate Impact', fontsize=14)
axes[0].legend(loc='lower left', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Statistical Parity Difference
for method, metrics in results_summary.items():
    if metrics['stat_parity_diff'] is not None:
        # === INICIO SOLUCIÓN ===

        # === FIN SOLUCIÓN ===

axes[1].axvline(x=0.1, color='red', linestyle='--', alpha=0.5,
                label='Max SPD (0.1)')
axes[1].set_xlabel('|Statistical Parity Difference|', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Trade-off: Accuracy vs Statistical Parity', fontsize=14)
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 5.2 Análisis de Métricas por Grupo

In [ ]:
# Comparación detallada de métricas por grupo racial
techniques = ['Baseline', 'Reweighing', 'ExpGrad-DP', 'ExpGrad-EO', 'ThresholdOpt']
predictions = [y_pred_baseline, y_pred_rw, y_pred_expgrad, y_pred_eo, y_pred_threshold]

# Calcular métricas por técnica y grupo
group_metrics = []

for tech, preds in zip(techniques, predictions):
    # African-American
    aa_mask = sensitive_test == 0
    aa_metrics = {
        'technique': tech,
        'group': 'African-American',
        'accuracy': accuracy_score(y_test[aa_mask], preds[aa_mask]),
        'precision': precision_score(y_test[aa_mask], preds[aa_mask], zero_division=0),
        'recall': recall_score(y_test[aa_mask], preds[aa_mask]),
        'selection_rate': preds[aa_mask].mean()
    }
    group_metrics.append(aa_metrics)

    # Caucasian
    c_mask = sensitive_test == 1
    c_metrics = {
        'technique': tech,
        'group': 'Caucasian',
        'accuracy': accuracy_score(y_test[c_mask], preds[c_mask]),
        'precision': precision_score(y_test[c_mask], preds[c_mask], zero_division=0),
        'recall': recall_score(y_test[c_mask], preds[c_mask]),
        'selection_rate': preds[c_mask].mean()
    }
    group_metrics.append(c_metrics)

group_metrics_df = pd.DataFrame(group_metrics)

# Visualización
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = ['accuracy', 'precision', 'recall', 'selection_rate']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]

    pivot_data = group_metrics_df.pivot(index='technique',
                                        columns='group',
                                        values=metric)
    pivot_data.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title(f'{metric.capitalize()} por Técnica y Grupo', fontsize=12)
    ax.set_xlabel('Técnica', fontsize=11)
    ax.set_ylabel(metric.capitalize(), fontsize=11)
    ax.legend(title='Grupo Racial')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 6. EJERCICIO 9

Basándote en todos los análisis, escribe tu recomendación. Completa este apartado con tu análisis y recomendación final.

#### 1. Property-Based Testing

- Técnica recomendada:
    > [Respuesta]


#### 2. Justificación

- ¿Por qué eligió esta técnica?
    > [Respuesta]

- ¿Qué compromisos acepta?
    > [Respuesta]

- ¿Qué métricas de equidad prioriza y por qué?
    > [Respuesta]

#### 3. Métricas clave de la solución recomendada

- Accuracy:
    > [Respuesta]

- Disparate Impact:
    > [Respuesta]

- Equal Opportunity Difference:
    > [Respuesta]


#### 4. Consideraciones éticas

- ¿Qué implicaciones tiene para los individuos afectados?
    > [Respuesta]

- ¿Cómo afecta a los diferentes grupos raciales?
    > [Respuesta]

- ¿Es aceptable el nivel de sesgo restante?
    > [Respuesta]


#### 5. Limitaciones y trabajo futuro
- ¿Qué limitaciones tiene su solución?
    > [Respuesta]

- ¿Qué análisis adicionales recomendaría?
    > [Respuesta]

## CONCLUSIONES

En esta práctica hemos:
1. Identificado sesgos significativos en el sistema COMPAS original
2. Aplicado múltiples técnicas de mitigación (pre, in, y post-processing)
3. Evaluado el compromiso entre accuracy y equidad
4. Seleccionado una solución que balancea rendimiento y justicia algorítmica

**Reflexión final**: El caso COMPAS ilustra la importancia crítica de auditar y mitigar sesgos en sistemas de IA que afectan la libertad y vida de las personas.